# **Imports and globals**


In [1]:
import numpy as np
import pandas as pd
import os
import glob
from colorama import Fore, Back, Style, init

In [2]:
# --- 1. Fixed Paths ---
OUTPUT_DIR = '../data/'
MOVIES_PER_GENRE_FOLDER = '../data/dataset/1_movies_per_genre'
# Point this to the actual folder containing the 1,150 review CSVs
REVIEWS_RAW_FOLDER = '../data/dataset/2_reviews_per_movie_raw'

# Output Filenames
METADATA_OUT = os.path.join(OUTPUT_DIR, 'merged_movies_per_genre.csv')
REVIEWS_OUT = os.path.join(OUTPUT_DIR, 'merged_reviews.csv')

# --- 2. Safety Check ---
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"📁 Created missing directory: {OUTPUT_DIR}")

# Merge CSVs

In [3]:
def process_and_merge(input_folder, output_path, label_column_name):
    files = glob.glob(os.path.join(input_folder, "*.csv"))
    if not files:
        print(f"❌ No files found in {input_folder}. Check your paths!")
        return None

    all_data = []
    print(f"--- Scanning {len(files)} files in {os.path.basename(input_folder)} ---")

    for f in files:
        df = pd.read_csv(f)
        file_name = os.path.basename(f)

        # Add source label (Genre name or Movie title)
        df[label_column_name] = file_name.replace(".csv", "")

        # --- NaN CHECKING ---
        if df.isnull().values.any():
            nan_rows = df[df.isnull().any(axis=1)]
            for idx, row in nan_rows.iterrows():
                missing_cols = row.index[row.isnull()].tolist()
                print(f"📍 NaN found in [{file_name}] | Row: {idx} | Columns: {missing_cols}")

        all_data.append(df)

    # Merge and Save
    final_df = pd.concat(all_data, ignore_index=True)
    final_df.to_csv(output_path, index=False)
    print(f"✅ Saved merged file to: {output_path}\n")
    return final_df

# --- 3. Run for both ---
df_meta = process_and_merge(MOVIES_PER_GENRE_FOLDER, METADATA_OUT, 'genre_label')
df_revs = process_and_merge(REVIEWS_RAW_FOLDER, REVIEWS_OUT, 'movie_title')

--- Scanning 17 files in 1_movies_per_genre ---
✅ Saved merged file to: ../data/merged_movies_per_genre.csv

--- Scanning 1150 files in 2_reviews_per_movie_raw ---
✅ Saved merged file to: ../data/merged_reviews.csv



# Find NaN

In [4]:
folders = {
    "Genres Metadata": MOVIES_PER_GENRE_FOLDER,
    "Movie Reviews": '../data/dataset/2_reviews_per_movie_raw'
}

nan_report_list = []
all_columns_found = set() # Using a set to keep only unique names

print("🚀 Scanning all CSVs...")

for category, folder_path in folders.items():
    files = glob.glob(os.path.join(folder_path, "*.csv"))

    for file_path in files:
        try:
            df = pd.read_csv(file_path)
            file_name = os.path.basename(file_path)

            # Record all columns found in this file
            all_columns_found.update(df.columns.tolist())

            # Detect NaNs and String "Nulls"
            nan_mask = df.isnull()
            string_null_mask = df.map(lambda x: str(x).strip().lower() in ['null', 'nan', ''])
            combined_mask = nan_mask | string_null_mask

            if combined_mask.values.any():
                rows_with_issues = df[combined_mask.any(axis=1)]
                for index, row in rows_with_issues.iterrows():
                    problem_cols = row.index[combined_mask.loc[index]].tolist()
                    nan_report_list.append({
                        "file": file_name,
                        "line": index + 2,
                        "columns": problem_cols
                    })
        except Exception:
            continue

print(f"✅ Done. Scanned all files. Unique columns stored.")

🚀 Scanning all CSVs...
✅ Done. Scanned all files. Unique columns stored.


## Display found NaNs && Nulls

In [5]:
nan_report_list = []
all_columns_found = set()

folders = {
    "Genres Metadata": '../data/dataset/1_movies_per_genre',
    "Movie Reviews": '../data/dataset/2_reviews_per_movie_raw'
}

print("🚀 Scanning... please wait.")

for category, folder_path in folders.items():
    files = glob.glob(os.path.join(folder_path, "*.csv"))
    for file_path in files:
        try:
            df = pd.read_csv(file_path)
            all_columns_found.update(df.columns.tolist())

            # Optimized check for NaNs and literal "Null" strings
            mask = df.isnull() | df.map(lambda x: str(x).strip().lower() in ['null', 'nan', ''])

            if mask.values.any():
                # Get the specific columns that triggered the mask in this file
                problem_cols_in_file = df.columns[mask.any()].tolist()

                # We store the issue to report it grouped by category later
                for index, row in df[mask.any(axis=1)].iterrows():
                    nan_report_list.append({
                        "category": category, # <--- Crucial for grouping
                        "file": os.path.basename(file_path),
                        "line": index + 2,
                        "columns": row.index[mask.loc[index]].tolist()
                    })
        except: continue
print(f"✅ Done! Found {len(all_columns_found)} columns and {len(nan_report_list)} issues.")

# 1. Logic to extract unique column names grouped by their folder category
grouped_cols_with_nans = {}

for entry in nan_report_list:
    cat = entry['category']
    cols = entry['columns']

    if cat not in grouped_cols_with_nans:
        grouped_cols_with_nans[cat] = set()

    grouped_cols_with_nans[cat].update(cols)

# 2. Display the Grouped Unique Columns
print(Style.BRIGHT + Fore.MAGENTA + "📋 UNIQUE COLUMNS WITH NaNs (GROUPED BY FOLDER)")
print(Fore.MAGENTA + "=" * 65)

for category, columns in grouped_cols_with_nans.items():
    print(f"\n{Fore.CYAN}📁 Folder: {category}")
    print(f"{Fore.WHITE}" + "-" * 35)

    for i, col in enumerate(sorted(list(columns)), 1):
        # Highlighting the specific feature name that has 'holes' in it
        print(f"{Fore.WHITE}{i: >2}. {Fore.YELLOW}{col}")

print(f"\n{Style.BRIGHT}{Fore.GREEN}Check complete. Total of {sum(len(v) for v in grouped_cols_with_nans.values())} problematic features identified.")

🚀 Scanning... please wait.
✅ Done! Found 16 columns and 96181 issues.
📋 UNIQUE COLUMNS WITH NaNs (GROUPED BY FOLDER)

📁 Folder: Genres Metadata
-----------------------------------
 1. movie_rated

📁 Folder: Movie Reviews
-----------------------------------
 1. rating
 2. title

Check complete. Total of 3 problematic features identified.


In [6]:
from colorama import Fore, Style

# 1. Structure: Category -> Column Name -> { Filename: {Lines} }
detailed_map = {}

for entry in nan_report_list:
    cat = entry['category']
    col_list = entry['columns']
    fname = entry['file']
    line = entry['line']

    if cat not in detailed_map:
        detailed_map[cat] = {}

    for col in col_list:
        if col not in detailed_map[cat]:
            detailed_map[cat][col] = {}

        if fname not in detailed_map[cat][col]:
            detailed_map[cat][col][fname] = set()

        detailed_map[cat][col][fname].add(line)

# 2. Display the Detailed Report
print(Style.BRIGHT + Fore.MAGENTA + "📋 COLUMN-CENTRIC ISSUE REPORT (WITH CSV NAMES)")
print(Fore.MAGENTA + "=" * 100)

for category, columns in detailed_map.items():
    print(f"\n{Back.CYAN}{Fore.BLACK} 📁 FOLDER: {category} {Style.RESET_ALL}")

    for col_name in sorted(columns.keys()):
        print(f"\n  {Fore.YELLOW}🔹 Column: {col_name}")
        print(f"  " + "-" * 30)

        # Sort files alphabetically
        for fname in sorted(columns[col_name].keys()):
            # Sort lines numerically
            lines = sorted(list(columns[col_name][fname]))
            line_str = ", ".join(map(str, lines))

            print(f"    {Fore.CYAN}{fname:<35} {Fore.WHITE}| {Fore.RED}Lines: {line_str}")

print(f"\n{Style.BRIGHT}{Fore.GREEN}✅ Report Complete. {len(nan_report_list)} total issues mapped.")

📋 COLUMN-CENTRIC ISSUE REPORT (WITH CSV NAMES)

 📁 FOLDER: Genres Metadata 

  🔹 Column: movie_rated
  ------------------------------
    War.csv                             | Lines: 32

 📁 FOLDER: Movie Reviews 

  🔹 Column: rating
  ------------------------------
    10 Cloverfield Lane 2016.csv        | Lines: 16, 65, 67, 134, 166, 183, 188, 213, 296, 356, 399, 404, 420, 438, 450, 462, 465, 498, 558, 594, 602, 626, 631, 672
    10 Things I Hate About You 1999.csv | Lines: 2, 7, 11, 12, 14, 25, 27, 33, 35, 48, 63, 102, 109, 115, 121, 122, 128, 140, 148, 153, 155, 158, 163, 166, 203, 204, 221, 232, 246, 269, 271, 273, 278, 281, 289, 292, 293, 296, 298, 301, 302, 305, 308, 311, 312, 316, 319, 320, 323, 324, 325, 327, 333, 339, 340, 341, 342, 343, 346, 348, 350, 353, 354, 357, 361, 363, 364, 366, 371, 372, 373, 375, 376, 377, 378, 379, 383, 385, 388, 393, 397, 398, 399, 404, 405, 406, 407, 408, 413, 414, 415, 416, 421, 423, 426, 428, 431, 437, 439, 441, 446, 447, 450, 452, 453, 455, 461